In [1]:
import warnings
warnings.filterwarnings('ignore')
from yfinance import download
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Análise de Preço - Ativo Individual

Dashboard de análise técnica e estatística para um ativo específico.

**Indicadores incluídos:**
- Preço vs Regressão Linear (tendência de longo prazo)
- Médias Móveis (200 dias e acumulada)
- Desvio percentual em relação à tendência
- Stochastic %K (momentum de curto prazo)
- Métricas de risco e retorno

In [2]:
# "USDBRL=X"

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURAÇÃO
# ══════════════════════════════════════════════════════════════
param = {
    'ticker': '^BVSP',
    'interval': '1d',
    'period': '10y',
    'column': 'Close',
    'ma_period': 200,        # Período da média móvel
    'stoch_period': 8,       # Período do Stochastic %K
}

# ══════════════════════════════════════════════════════════════
# DOWNLOAD DOS DADOS
# ══════════════════════════════════════════════════════════════
df = download(
    param['ticker'], 
    period=param['period'], 
    interval=param['interval'], 
    progress=False, 
    auto_adjust=False
).droplevel(1, axis=1)

print(f"Dados carregados: {len(df)} registros de {df.index[0].strftime('%d/%m/%Y')} a {df.index[-1].strftime('%d/%m/%Y')}")

# ══════════════════════════════════════════════════════════════
# CÁLCULO DOS INDICADORES
# ══════════════════════════════════════════════════════════════
col = param['column']
x = df.index.astype(int)

# Regressão linear (calculada uma única vez)
coef = np.polyfit(x, df[col], 1)
df['reg_lin'] = coef[0] * x + coef[1]

# Médias móveis
df['mean_acc'] = df[col].expanding().mean()
df['mean_200'] = df[col].rolling(param['ma_period']).mean()

# Stochastic %K
stoch_period = param['stoch_period']
lowest = df['Low'].rolling(stoch_period).min()
highest = df['High'].rolling(stoch_period).max()
df['stoch_k'] = 100 * (df[col] - lowest) / (highest - lowest)

# Desvio percentual da regressão
df['dif_close_reg_lin'] = ((df[col] - df['reg_lin']) / df['reg_lin']) * 100

Dados carregados: 2485 registros de 24/02/2016 a 24/02/2026


In [4]:
# Cálculo de métricas essenciais
retornos = df[param['column']].pct_change().dropna()

# Retorno acumulado no período
retorno_total = (df[param['column']].iloc[-1] / df[param['column']].iloc[0] - 1) * 100

# Retorno anualizado (CAGR)
dias = len(df)
anos = dias / 252
cagr = ((df[param['column']].iloc[-1] / df[param['column']].iloc[0]) ** (1/anos) - 1) * 100

# Volatilidade anualizada
volatilidade = retornos.std() * np.sqrt(252) * 100

# Máximo Drawdown
preco_maximo = df[param['column']].expanding().max()
drawdown = (df[param['column']] - preco_maximo) / preco_maximo * 100
max_drawdown = drawdown.min()

# Percentil do preço atual
preco_atual = df[param['column']].iloc[-1]
percentil_preco = (df[param['column']] < preco_atual).sum() / len(df) * 100

# Percentil do desvio atual
desvio_atual = df['dif_close_reg_lin'].iloc[-1]
percentil_desvio = (df['dif_close_reg_lin'].dropna() < desvio_atual).sum() / len(df['dif_close_reg_lin'].dropna()) * 100

# R² da regressão linear
from scipy.stats import linregress
slope, intercept, r_value, p_value, std_err = linregress(
    np.arange(len(df)), df[param['column']]
)
r2 = r_value ** 2

# Slope anualizado (% ao ano)
slope_anual = (slope * 252 / df[param['column']].mean()) * 100

# Sharpe Ratio simplificado (assumindo taxa livre de risco = 0)
sharpe = (retornos.mean() * 252) / (retornos.std() * np.sqrt(252))

metricas = {
    'Preço Atual': f"R$ {preco_atual:.2f}",
    'Retorno Total': f"{retorno_total:.1f}%",
    'CAGR': f"{cagr:.1f}% a.a.",
    'Volatilidade': f"{volatilidade:.1f}% a.a.",
    'Max Drawdown': f"{max_drawdown:.1f}%",
    'Sharpe Ratio': f"{sharpe:.2f}",
    'R² Regressão': f"{r2:.2%}",
    'Tendência': f"{slope_anual:.1f}% a.a.",
    'Percentil Preço': f"{percentil_preco:.0f}%",
    'Percentil Desvio': f"{percentil_desvio:.0f}%",
    'Desvio Atual': f"{desvio_atual:.1f}%"
}

print(f"{'='*50}")
print(f"  RESUMO - {param['ticker'].upper()}")
print(f"{'='*50}")
for k, v in metricas.items():
    print(f"  {k:.<20} {v:>15}")
print(f"{'='*50}")

  RESUMO - ^BVSP
  Preço Atual.........    R$ 189966.09
  Retorno Total.......          351.4%
  CAGR................      16.5% a.a.
  Volatilidade........      23.1% a.a.
  Max Drawdown........          -46.8%
  Sharpe Ratio........            0.78
  R² Regressão........          84.67%
  Tendência...........       8.5% a.a.
  Percentil Preço.....            100%
  Percentil Desvio....            100%
  Desvio Atual........           30.3%


## Interpretação das Métricas

| Métrica | Descrição |
|---------|-----------|
| **CAGR** | Retorno anualizado composto (crescimento médio por ano) |
| **Volatilidade** | Risco medido pelo desvio padrão dos retornos (anualizado) |
| **Max Drawdown** | Maior queda do pico ao vale no período |
| **Sharpe Ratio** | Retorno por unidade de risco (>1 = bom, >2 = excelente) |
| **R² Regressão** | Qualidade do ajuste linear (>0.8 = tendência forte) |
| **Tendência** | Inclinação da regressão em % ao ano |
| **Percentil Preço** | Posição do preço atual no histórico (50% = mediana) |
| **Percentil Desvio** | Posição do desvio atual na distribuição histórica |

In [5]:
df.tail(3)

Price,Adj Close,Close,High,Low,Open,Volume,reg_lin,mean_acc,mean_200,stoch_k,dif_close_reg_lin
Date,,,,,,,,,,,
2026-02-20,190534.00000,190534.00000,190727.00000,186700.000000,188531.000000,9988100,145746.687532,102871.434958,149176.600000,97.518323,30.729558
2026-02-23,188854.00000,188854.00000,191003.00000,188526.000000,190532.000000,9901300,145817.176973,102906.049517,149453.880000,70.726059,29.514234
2026-02-24,189966.09375,189966.09375,189966.09375,188854.453125,188854.453125,0,145840.673453,102941.083740,149722.550469,85.875136,30.255908


In [6]:
vals_dif_close_reg_lin = df['dif_close_reg_lin'].dropna()
valor_atual_dif_close_reg_lin = vals_dif_close_reg_lin.iloc[-1]
media_dif_close_reg_lin = vals_dif_close_reg_lin.mean()
q25_dif_close_reg_lin = vals_dif_close_reg_lin.quantile(0.25)
q75_dif_close_reg_lin = vals_dif_close_reg_lin.quantile(0.75)
std_desvio = vals_dif_close_reg_lin.std()

# Paleta de cores profissional
CORES = {
    'principal': '#1a1a2e',      # Azul escuro quase preto
    'tendencia': '#6c5ce7',      # Roxo elegante
    'ma200': '#00cec9',          # Turquesa
    'media_acc': '#fd79a8',      # Rosa
    'positivo': '#00b894',       # Verde
    'negativo': '#d63031',       # Vermelho
    'neutro': '#636e72',         # Cinza
    'fundo_hist': '#74b9ff',     # Azul claro
    'drawdown': '#e17055',       # Laranja avermelhado
}

# FIGURA
fig = make_subplots(
    rows=4,
    cols=2,
    shared_xaxes=True,
    vertical_spacing=0.04,
    horizontal_spacing=0.03,
    column_widths=[0.72, 0.28],
    row_heights=[0.42, 0.17, 0.24, 0.17],
    specs=[
        [{"colspan": 2}, None],
        [{"colspan": 2}, None],
        [{}, {}],
        [{"colspan": 2}, None]
    ],
)

# ══════════════════════════════════════════════════════════════
# GRÁFICO 1 — PREÇO
# ══════════════════════════════════════════════════════════════
fig.add_trace(go.Scatter(
    x=df.index,
    y=df[param['column']],
    name='Preço',
    line=dict(color=CORES['principal'], width=2.5),
    hovertemplate='R$ %{y:.2f}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['reg_lin'],
    name='Tendência',
    line=dict(color=CORES['tendencia'], width=2, dash='dot'),
    hovertemplate='R$ %{y:.2f}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['mean_200'],
    name='MA 200',
    line=dict(color=CORES['ma200'], width=1.5),
    hovertemplate='R$ %{y:.2f}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['mean_acc'],
    name='Média Histórica',
    line=dict(color=CORES['media_acc'], width=1, dash='dash'),
    opacity=0.7,
    hovertemplate='R$ %{y:.2f}<extra></extra>'
), row=1, col=1)

# # Anotação do preço atual
# fig.add_annotation(
#     x=df.index[-1],
#     y=preco_atual,
#     text=f"<b>R$ {preco_atual:.2f}</b>",
#     showarrow=True,
#     arrowhead=0,
#     ax=50,
#     ay=0,
#     font=dict(size=11, color=CORES['principal']),
#     bgcolor='white',
#     bordercolor=CORES['principal'],
#     borderwidth=1,
#     borderpad=4,
#     row=1, col=1
# )

# ══════════════════════════════════════════════════════════════
# GRÁFICO 2 — DRAWDOWN
# ══════════════════════════════════════════════════════════════
fig.add_trace(go.Scatter(
    x=df.index,
    y=drawdown,
    name='Drawdown',
    fill='tozeroy',
    line=dict(color=CORES['drawdown'], width=1),
    fillcolor='rgba(225, 112, 85, 0.4)',
    showlegend=False,
    hovertemplate='%{y:.1f}%<extra></extra>'
), row=2, col=1)

fig.add_hline(
    y=max_drawdown,
    row=2, col=1,
    line_color=CORES['negativo'],
    line_dash='dash',
    line_width=1.5,
)

fig.add_annotation(
    x=df.index[0],
    y=max_drawdown,
    text=f"  Max Drawdown: {max_drawdown:.1f}%",
    showarrow=False,
    xanchor='left',
    font=dict(size=10, color=CORES['negativo']),
    row=2, col=1
)

# ══════════════════════════════════════════════════════════════
# GRÁFICO 3 — DESVIO DA REGRESSÃO
# ══════════════════════════════════════════════════════════════

# Bandas de desvio padrão (da mais externa para mais interna para layering correto)
bandas = [
    (3, 'rgba(214, 48, 49, 0.15)', CORES['negativo']),   # Vermelho
    (2, 'rgba(253, 121, 168, 0.18)', CORES['media_acc']), # Rosa
    (1, 'rgba(108, 92, 231, 0.20)', CORES['tendencia']),  # Roxo
]

# Bandas no gráfico de linha (hrect)
for n_sigma, cor_fill, cor_linha in bandas:
    fig.add_hrect(
        y0=media_dif_close_reg_lin - n_sigma * std_desvio,
        y1=media_dif_close_reg_lin + n_sigma * std_desvio,
        row=3, col=1,
        fillcolor=cor_fill,
        line_width=0,
        layer='below',
    )

# Bandas no histograma (vrect)
for n_sigma, cor_fill, cor_linha in bandas:
    fig.add_vrect(
        x0=media_dif_close_reg_lin - n_sigma * std_desvio,
        x1=media_dif_close_reg_lin + n_sigma * std_desvio,
        row=3, col=2,
        fillcolor=cor_fill,
        line_width=0,
        layer='below',
    )

# Linhas das bandas no gráfico de linha (usando shapes para maior controle)
for n_sigma, cor_fill, cor_linha in bandas:
    for y_val in [media_dif_close_reg_lin + n_sigma * std_desvio, 
                  media_dif_close_reg_lin - n_sigma * std_desvio]:
        fig.add_shape(
            type='line',
            x0=df.index[0], x1=df.index[-1],
            y0=y_val, y1=y_val,
            xref='x3', yref='y3',
            line=dict(color=cor_linha, width=1.5, dash='dash'),
        )

# Linhas das bandas no histograma (usando shapes)
hist_min = vals_dif_close_reg_lin.min()
hist_max = vals_dif_close_reg_lin.max()
for n_sigma, cor_fill, cor_linha in bandas:
    for x_val in [media_dif_close_reg_lin + n_sigma * std_desvio,
                  media_dif_close_reg_lin - n_sigma * std_desvio]:
        fig.add_shape(
            type='line',
            x0=x_val, x1=x_val,
            y0=0, y1=1,
            xref='x4', yref='y4 domain',
            line=dict(color=cor_linha, width=1.5, dash='dash'),
        )

# Anotações das bandas (lado direito do gráfico de linha)
for n_sigma, label in [(1, '±1σ'), (2, '±2σ'), (3, '±3σ')]:
    fig.add_annotation(
        x=1.01, xref='x3 domain',
        y=media_dif_close_reg_lin + n_sigma * std_desvio,
        text=f"{media_dif_close_reg_lin + n_sigma * std_desvio:.1f}%",
        showarrow=False,
        font=dict(size=8, color=CORES['neutro']),
        xanchor='left',
        row=3, col=1
    )
    fig.add_annotation(
        x=1.01, xref='x3 domain',
        y=media_dif_close_reg_lin - n_sigma * std_desvio,
        text=f"{media_dif_close_reg_lin - n_sigma * std_desvio:.1f}%",
        showarrow=False,
        font=dict(size=8, color=CORES['neutro']),
        xanchor='left',
        row=3, col=1
    )

# Área preenchida para desvio
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['dif_close_reg_lin'],
    name='Desvio',
    fill='tozeroy',
    line=dict(color=CORES['tendencia'], width=1.5),
    fillcolor='rgba(108, 92, 231, 0.15)',
    showlegend=False,
    hovertemplate='%{y:.1f}%<extra></extra>'
), row=3, col=1)

# Linha zero (média da regressão)
fig.add_hline(y=0, row=3, col=1, line_color=CORES['neutro'], line_width=1.5)

# HISTOGRAMA
cor_hist = CORES['positivo'] if valor_atual_dif_close_reg_lin > 0 else CORES['negativo']

fig.add_trace(go.Histogram(
    x=vals_dif_close_reg_lin,
    nbinsx=50,
    opacity=0.85,
    marker=dict(
        color=CORES['fundo_hist'],
        line=dict(color='white', width=0.5)
    ),
    showlegend=False,
    hovertemplate='%{x:.1f}%<br>Freq: %{y}<extra></extra>'
), row=3, col=2)

fig.update_yaxes(showticklabels=False, row=3, col=2)

# Linha zero no histograma
fig.add_vline(x=0, row=3, col=2, line_color=CORES['neutro'], line_width=1.5)

# Linha do valor atual no histograma
fig.add_vline(x=valor_atual_dif_close_reg_lin, row=3, col=2, 
              line_color=cor_hist, line_width=3)

# Anotação do valor atual no histograma
fig.add_annotation(
    x=valor_atual_dif_close_reg_lin,
    y=1.05,
    yref='y4 domain',
    text=f"<b>{valor_atual_dif_close_reg_lin:+.1f}%</b>",
    showarrow=False,
    font=dict(size=12, color=cor_hist),
    bgcolor='white',
    row=3, col=2
)

# ══════════════════════════════════════════════════════════════
# GRÁFICO 4 — STOCHASTIC
# ══════════════════════════════════════════════════════════════
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['stoch_k'],
    name='Stoch %K',
    line=dict(color=CORES['principal'], width=1.2),
    showlegend=False,
    hovertemplate='%{y:.1f}<extra></extra>'
), row=4, col=1)

# Zonas coloridas
fig.add_hrect(y0=0, y1=20, row=4, col=1, 
              fillcolor='rgba(0, 184, 148, 0.15)', line_width=0, layer='below')
fig.add_hrect(y0=80, y1=100, row=4, col=1, 
              fillcolor='rgba(214, 48, 49, 0.15)', line_width=0, layer='below')

fig.add_hline(y=20, row=4, col=1, line_dash='dash', line_width=1, line_color=CORES['positivo'])
fig.add_hline(y=80, row=4, col=1, line_dash='dash', line_width=1, line_color=CORES['negativo'])
fig.add_hline(y=50, row=4, col=1, line_dash='dot', line_width=0.5, line_color=CORES['neutro'])

# Valor atual do Stochastic
stoch_atual = df['stoch_k'].iloc[-1]
cor_stoch = CORES['positivo'] if stoch_atual < 30 else (CORES['negativo'] if stoch_atual > 70 else CORES['neutro'])
fig.add_annotation(
    x=df.index[-1],
    y=stoch_atual,
    text=f"<b>{stoch_atual:.0f}</b>",
    showarrow=True,
    arrowhead=0,
    ax=35,
    ay=0,
    font=dict(size=10, color=cor_stoch),
    bgcolor='white',
    bordercolor=cor_stoch,
    borderwidth=1,
    row=4, col=1
)

# ══════════════════════════════════════════════════════════════
# LAYOUT FINAL
# ══════════════════════════════════════════════════════════════
fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    height=1000,
    width=1400,
    margin=dict(l=60, r=60, t=80, b=40),
    
    # Título principal
    title=dict(
        text=f"<b>{param['ticker'].upper()}</b> — Análise de Preço",
        font=dict(size=20, color=CORES['principal']),
        x=0.5,
        xanchor='center'
    ),
    
    # Legenda
    legend=dict(
        orientation='h',
        y=1.02,
        x=0.5,
        xanchor='center',
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor=CORES['neutro'],
        borderwidth=0.5,
        font=dict(size=10)
    ),
    
    # Fonte global
    font=dict(family='Segoe UI, Arial', size=10, color=CORES['principal'])
)

# Títulos dos eixos Y
fig.update_yaxes(title_text='Preço (R$)', title_font=dict(size=10), row=1, col=1)
fig.update_yaxes(title_text='DD (%)', title_font=dict(size=10), row=2, col=1)
fig.update_yaxes(title_text='Desvio (%)', title_font=dict(size=10), row=3, col=1)
fig.update_yaxes(title_text='%K', title_font=dict(size=10), row=4, col=1)

# Limites do Stochastic
fig.update_yaxes(range=[0, 100], row=4, col=1)

# Grid mais sutil
fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.05)')
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.05)')

# Mostrar datas apenas no último gráfico
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_xaxes(showticklabels=False, row=2, col=1)
fig.update_xaxes(showticklabels=False, row=3, col=1)
fig.update_xaxes(showticklabels=True, row=4, col=1, tickformat='%b/%Y')

fig.show()